In [1]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.multioutput import MultiOutputClassifier
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from skmultilearn.adapt import MLkNN
from scipy.sparse import csr_matrix
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import classification_report, accuracy_score
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.feature_selection import SelectKBest, chi2



#Carga de datos

train_data = pd.read_csv("../../Data/train_indexado.csv")
test_data = pd.read_csv("../../Data/test_indexado.csv")

# Definir las clases de emociones
emotion_classes = train_data.columns[2:].tolist()




In [2]:
import re
from nltk.stem import WordNetLemmatizer

# --- PREPROCESAMIENTO PARA LEMATIZACIÓN ---
lemmatizer = WordNetLemmatizer()

def preprocess_text(text):
    words = re.findall(r'\b\w+\b', text.lower())
    lemmatized_words = [lemmatizer.lemmatize(word) for word in words]
    return " ".join(lemmatized_words)

X_train_lem = train_data['Text'].apply(preprocess_text)
X_test_lem = test_data['Text'].apply(preprocess_text)

In [3]:
# TF-IDF VECTORIZACIÓN
vectorizer = TfidfVectorizer(lowercase=True, strip_accents="unicode", max_features=10000)
X_train = vectorizer.fit_transform(X_train_lem)
X_test = vectorizer.transform(X_test_lem)
y_train = np.asarray(train_data[emotion_classes])
y_test = np.asarray(test_data[emotion_classes])

In [4]:
for k in [500, 1000, 2000, 3000, 5000]:
    print(f"\n*** Evaluación con Chi-cuadrada (Número de atributos: {k} ) ***")

    # Selección de features/características
    selector = SelectKBest(score_func=chi2, k=k)
    X_train_chi = selector.fit_transform(X_train, y_train)
    X_test_chi = selector.transform(X_test)

    # SVM
    svm_clf = SVC(kernel='linear', class_weight='balanced')
    multi_smv = MultiOutputClassifier(svm_clf)
    multi_smv.fit(X_train_chi, y_train)

    y_pred_svm = multi_smv.predict(X_test_chi)

        # Reporte SVM
    report_dict_svm = classification_report(y_test, y_pred_svm, output_dict=True, zero_division=0)

    f1_macro_svm = report_dict_svm["macro avg"]["f1-score"]
    recall_macro_svm = report_dict_svm["macro avg"]["recall"]

    print("\nResultados SVM Multilabel:\n")
    print("Accuracy:", accuracy_score(y_test, y_pred_svm))
    print(f"F1 Score (macro avg): {f1_macro_svm:.5f}")
    print(f"Recall Score (macro avg): {recall_macro_svm:.5f}")

    # Random Forest

    rf_clf =RandomForestClassifier(random_state=42)
    multi_rf = MultiOutputClassifier(rf_clf)
    multi_rf.fit(X_train_chi, y_train)

    y_pred_rf= multi_rf.predict(X_test_chi)

        # Reporte RF
    report_dict_rf = classification_report(y_test, y_pred_rf, output_dict=True, zero_division=0)

    f1_macro_rf = report_dict_rf["macro avg"]["f1-score"]
    recall_macro_rf = report_dict_rf["macro avg"]["recall"]

    print("\nResultados Random Forest:\n")
    print("Accuracy:", accuracy_score(y_test, y_pred_rf))
    print(f"F1 Score (macro avg): {f1_macro_rf:.5f}")
    print(f"Recall Score (macro avg): {recall_macro_rf:.5f}")

    # MLkNN
    mlknn = MLkNN(k=3)
    mlknn.fit(X_train_chi, csr_matrix(y_train))
    y_pred_mlknn = mlknn.predict(X_test_chi)

        # Reporte MLKNN 
    report_dict_mlknn = classification_report(y_test, y_pred_mlknn, output_dict=True, zero_division=0)
    f1_macro_mlknn = report_dict_mlknn["macro avg"]["f1-score"]
    recall_macro_mlknn = report_dict_mlknn["macro avg"]["recall"]
    print("\nResultados MLkNN\n")
    print("Accuracy:", accuracy_score(y_test, y_pred_mlknn))
    print(f"F1 Score (macro avg): {f1_macro_mlknn:.5f}")
    print(f"Recall Score (macro avg): {recall_macro_mlknn:.5f}")



    







*** Evaluación con Chi-cuadrada (Número de atributos: 500 ) ***

Resultados SVM Multilabel:

Accuracy: 0.11369080523309379
F1 Score (macro avg): 0.31468
Recall Score (macro avg): 0.70171

Resultados Random Forest:

Accuracy: 0.326147042564953
F1 Score (macro avg): 0.32095
Recall Score (macro avg): 0.25938

Resultados MLkNN

Accuracy: 0.33001658374792703
F1 Score (macro avg): 0.30450
Recall Score (macro avg): 0.23714

*** Evaluación con Chi-cuadrada (Número de atributos: 1000 ) ***

Resultados SVM Multilabel:

Accuracy: 0.10539893126957804
F1 Score (macro avg): 0.32508
Recall Score (macro avg): 0.70944

Resultados Random Forest:

Accuracy: 0.3161967938087341
F1 Score (macro avg): 0.32013
Recall Score (macro avg): 0.25229

Resultados MLkNN

Accuracy: 0.32135618205269945
F1 Score (macro avg): 0.26145
Recall Score (macro avg): 0.21955

*** Evaluación con Chi-cuadrada (Número de atributos: 2000 ) ***

Resultados SVM Multilabel:

Accuracy: 0.09415883545236779
F1 Score (macro avg): 0.34401
R